In [2]:
from resources.imports import *

from resources.calculations import smooth, get_nodes, get_struts, get_ductileData, get_fractureData

In [3]:
### GLOBAL PATH INPUTS

pData = 'data/'

pAl          = pData + 'Al/'
pAK          = pAl + 'AK/'
pUTdisNodes  = pAK + 'Ductile-disNodes-FCC-12X16/'
pUTdisNodes2 = pAK + '20_RD02_10mm/'
pUTdisStruts = pAK + 'Ductile-disStruts-FCC-12X16/'
pFTdisNodes  = pAK + 'Fracture-disNodes/'

In [4]:
runMATLAB   = False
pMATLAB     = pUTdisNodes

runINOUT_AK = False
pINOUT_AK   = pUTdisNodes2

# MATLAB to CSV Data File Conversion

In [5]:
### INPUT

mode = 'Ductile'
dis = 'disNodes'

MATin  = pMATLAB + 'raw/' + 'inputData-20-frame-0.mat'
MATout = pMATLAB + 'raw/' + 'outputStressStrain-20-1-1500.mat'

CSVin  = pMATLAB + f'{mode}-{dis}-IN.csv'
CSVout = pMATLAB + f'{mode}-{dis}-OUT.csv'
append = False

In [6]:
def mat_to_csv(MATin, MATout, CSVin, CSVout, append=False):
    input_mat = scipy.io.loadmat(MATin)
    in_data = pd.DataFrame(list(input_mat.values())[3])
    in_data = in_data.rename(columns={i:str(i) for i in range(len(in_data.columns))})
    if append:
        in_header = pd.read_csv(CSVin, index_col=0)
        in_header = in_header.rename(columns={i:str(i) for i in range(len(in_header.columns))})
        in_data = in_data + in_header.iloc[0].values
        in_data = pd.concat([in_header, in_data], ignore_index=True)
    in_data.to_csv(CSVin)

    output_mat = scipy.io.loadmat(MATout)
    out_data = pd.DataFrame(list(output_mat.values())[3])
    out_data = out_data.rename(columns={i:str(i) for i in range(len(out_data.columns))})
    if append:
        out_header = pd.read_csv(CSVout, index_col=0)
        out_header = out_header.drop(out_header.columns[201:], axis=1)
        out_header = out_header.rename(columns={i:str(i) for i in range(len(out_header.columns))})
        out_data = pd.concat([out_header, out_data], ignore_index=True)
    out_data.to_csv(CSVout)

In [7]:
if runMATLAB:
    mat_to_csv(MATin, MATout, CSVin, CSVout, append=append)

# Create Input and Output CSV from Akash Raw .txt Files

### Inputs

In [8]:
def create_inputCSV_AK(directory):
    duct_disNodes = []
    
    pathRaw = directory + 'raw/'
    for ffile in os.listdir(pathRaw):
        num = int(ffile.split("-")[-1][:-4])
        if 'per' in ffile and ffile.endswith('.inp'):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + ffile, lineStart=17, lineEnd=430)
            duct_disNodes.insert(0, np.insert(nodes_df.to_numpy().flatten(), 0, 0))
        elif "dis" in ffile and ffile.endswith('.inp'):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + ffile, lineStart=17, lineEnd=430)
            duct_disNodes.append(np.insert(nodes_df.to_numpy().flatten(), 0, num))
    
    UTdisNodesIN_df = pd.DataFrame(duct_disNodes)
    UTdisNodesIN_df.to_csv(directory + 'Ductile-disNodes-IN.csv')
    return UTdisNodesIN_df

### Outputs

In [9]:
def create_outputCSV_AK(directory):
    duct_disNodes = []
    
    pathRaw = directory + 'raw/'
    for ffile in os.listdir(pathRaw):
        num = int(ffile.split("-")[-1][:-4])
        if 'per' in ffile and ffile.endswith('.txt'):
            output_df = get_ductileData(pathRaw + ffile, crit=0.2, delimiter=' ')
            duct_disNodes.insert(0, np.insert(output_df.x.tolist(), 0, 0))
            duct_disNodes.insert(1, np.insert(output_df.y_sm.tolist(), 0, 0))
        elif "dis" in ffile and ffile.endswith('.txt'):
            output_df = get_ductileData(pathRaw + ffile, crit=0.2, delimiter=' ')
            duct_disNodes.append(np.insert(output_df.y_sm.tolist(), 0, num))
    
    UTdisNodesOUT_df = pd.DataFrame(duct_disNodes, index=[int(i[0]) for i in duct_disNodes])
    
    UTdisNodes_drop = [i for i,j in zip([int(i[0]) for i in duct_disNodes], UTdisNodesOUT_df.isnull().any(axis=1)) if j == True]
    UTdisNodes_drop = UTdisNodes_drop + [i for i,j in zip([int(i[0]) for i in duct_disNodes], [int(i[1]) for i in duct_disNodes]) if j == 0.0]
    UTdisNodesOUT_df = UTdisNodesOUT_df.drop(UTdisNodes_drop, axis=0).drop(0, axis=1)
    UTdisNodesOUT_df.columns = range(UTdisNodesOUT_df.columns.size)
    UTdisNodesIN_df = pd.read_csv(directory + 'Ductile-disNodes-IN.csv', index_col=0)
    UTdisNodesIN_df.index = [int(i[0]) for i in duct_disNodes][1:]
    UTdisNodesIN_df = UTdisNodesIN_df.drop(UTdisNodes_drop, axis=0).drop('0', axis=1)
    UTdisNodesIN_df.columns = range(UTdisNodesIN_df.columns.size)
    UTdisNodesOUT_df.to_csv(directory + 'Ductile-disNodes-OUT.csv')
    UTdisNodesIN_df.to_csv(directory + 'Ductile-disNodes-IN.csv')
    
    return UTdisNodesOUT_df, UTdisNodesIN_df

In [10]:
if runINOUT_AK:
    UTdisNodesIN_df = create_inputCSV_AK(pINOUT_AK)
    UTdisNodesOUT_df, UTdisNodesIN_df = create_outputCSV_AK(pINOUT_AK)